In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import find_peaks, spectrogram
from TIF import Info, Anotaciones, RawSignal

In [ ]:
class ECGSignal(RawSignal):
    def __init__(self, data, sfreq, info=None, anotaciones=None, first_samp=0):
        super().__init__(data, sfreq, info, anotaciones, first_samp)
        self.r_peaks = []
        self.hr = 0.0

    def detectar_r_peaks(self, canal=0):
        signal = self.data[canal]
        peaks, _ = find_peaks(signal, distance=self.sfreq/2.5, height=np.mean(signal))
        self.r_peaks = peaks
        return peaks

    def calcular_hr(self):
        if not self.r_peaks:
            self.detectar_r_peaks()
        rr_intervals = np.diff(self.r_peaks) / self.sfreq
        self.hr = 60. / np.mean(rr_intervals)
        return self.hr

    def tiempo_frecuencia(self, canal):
        f, t, Sxx = spectrogram(self.data[canal], fs=self.sfreq)
        plt.pcolormesh(t, f, Sxx, shading='gouraud')
        plt.title(f"ECG Tiempo-Frecuencia - Canal {canal}")
        plt.ylabel('Frecuencia [Hz]')
        plt.xlabel('Tiempo [s]')
        plt.colorbar()
        plt.show()